In [4]:
#importing necessary libraries to perform nltk techniques
import re
import pandas as pd
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\adity\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\adity\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\adity\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [5]:
#function to clean the scentence 
def clean_text(sentence):
    # Keep only capital and small letters
    sentence = re.sub("[^a-zA-Z]", " ", sentence)

    # Convert to lowercase
    sentence = sentence.lower()

    # Split sentence into words
    words = sentence.split()

    # Remove stopwords and apply lemmatization
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    # Join words back into sentence
    sentence = " ".join(words)

    return sentence

In [ ]:
#cleaning all the titles and storing it inside cleaned_titles
df=pd.read_csv('reqfiles/AfterFE.csv')
df.shape
cleaned_titles = []

for title in df["title"]:
    print(title)
    cleaned_titles.append(clean_text(title))

In [7]:
#cleaning all the texts and storing it inside cleaned_texts
cleaned_texts = []

for text in df["text"]:
    cleaned_texts.append(clean_text(text))

In [8]:
cleaned_titles

['ben stein call th circuit court committed coup tat constitution',
 'trump drop steve bannon national security council',
 'puerto rico expects u lift jones act shipping restriction',
 'oops trump accidentally confirmed leaked israeli intelligence russia video',
 'donald trump head scotland reopen golf resort',
 'paul ryan responds dem sit gun control disgusting way video',
 'awesome diamond silk rip press believe video',
 'stand cheer ukip party leader slam germany france eu invasion phony refugee video',
 'north korea show sign serious talking u official',
 'trump signal willingness raise u minimum wage',
 'new jersey christie mull run lead republican party report',
 'france germany want iran reverse ballistic missile program',
 'aide eu commission head tweet picture white smoke brexit meeting may',
 'trump issue warning man army could isi video',
 'u give lao extra million help clear unexploded ordnance',
 'judge declares baby name illegal prevent emotional harm',
 'paul ryan take m

In [9]:
cleaned_texts

['st century wire say ben stein reputable professor pepperdine university also hollywood fame appearing tv show film ferris bueller day made provocative statement judge jeanine pirro show recently discussing halt imposed president trump executive order travel stein referred judgement th circuit court washington state coup tat executive branch constitution stein went call judge seattle political puppet judiciary political pawn watch interview complete statement note stark contrast rhetoric leftist medium pundit neglect note court ever blocked presidential order immigration past discus legal efficacy halt actual text executive order read trump news st century wire trump filessupport work subscribing becoming member wire tv',
 'washington reuters u president donald trump removed chief strategist steve bannon national security council wednesday reversing controversial decision early year give political adviser unprecedented role security discussion trump overhaul nsc confirmed white house 

In [10]:
print(len(cleaned_texts))
print(len(cleaned_titles))

44267
44267


In [11]:
from gensim.models import Word2Vec
import numpy as np
import pandas as pd

In [12]:
tokenized_titles = [sentence.split() for sentence in cleaned_titles]
tokenized_texts = [sentence.split() for sentence in cleaned_texts]

In [13]:
all_sentences = tokenized_titles + tokenized_texts

w2v_model = Word2Vec(
    sentences=all_sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

In [14]:
def average_word2vec(words, model, vector_size=100):
    valid_words = [word for word in words if word in model.wv]

    if len(valid_words) == 0:
        return np.zeros(vector_size)

    return np.mean([model.wv[word] for word in valid_words], axis=0)

In [15]:
title_vectors = [
    average_word2vec(words, w2v_model, 100)
    for words in tokenized_titles
]

text_vectors = [
    average_word2vec(words, w2v_model, 100)
    for words in tokenized_texts
]

In [16]:
title_vector_df = pd.DataFrame(
    title_vectors,
    columns=[f"title_w2v_{i}" for i in range(100)]
)

text_vector_df = pd.DataFrame(
    text_vectors,
    columns=[f"text_w2v_{i}" for i in range(100)]
)

In [17]:
df=df.drop(['title','text'],axis=1)
df=pd.concat([title_vector_df,text_vector_df,df.reset_index(drop=True)],axis=1)
df.shape


(44267, 201)

In [18]:
df.head()


,title_w2v_0,title_w2v_1,title_w2v_2,title_w2v_3,title_w2v_4,title_w2v_5,title_w2v_6,title_w2v_7,title_w2v_8,title_w2v_9,...,text_w2v_91,text_w2v_92,text_w2v_93,text_w2v_94,text_w2v_95,text_w2v_96,text_w2v_97,text_w2v_98,text_w2v_99,target
0,-0.440288,0.594304,0.488879,-1.093821,0.029279,-0.458521,-0.257205,-0.380895,0.117188,0.250903,...,-0.052323,0.322167,0.433010,0.083999,0.000530,-0.639745,0.915431,-0.684003,-0.080319,1
1,-1.181284,-0.981579,0.708629,-0.841638,-1.298389,0.469982,-0.484848,0.720773,0.044639,1.981418,...,0.919926,0.205002,-0.040219,-0.423374,-0.263470,-0.328516,0.902502,-0.356533,0.079089,0
2,-0.706966,-0.053229,-0.219723,0.301686,-0.626839,-0.811216,-0.297014,-0.227184,-0.075746,0.425133,...,0.497105,0.048756,-0.047975,0.312957,-0.282723,-0.189123,0.367328,-0.207796,-0.179578,0
3,-1.626629,-1.420631,0.133909,-0.571757,-0.742909,0.924996,-0.998800,-0.092274,1.210416,-0.092108,...,0.967439,-0.123410,0.065233,-0.249056,-0.078143,-0.754482,0.154473,-0.985923,0.690075,1
4,0.482687,-0.349060,1.028528,0.088326,-0.135576,0.583766,0.090581,-0.483990,0.196799,-0.229201,...,0.760256,0.111808,0.255053,0.063385,-0.060781,-0.141416,0.872366,-0.349999,-0.177563,0


In [19]:
df.to_csv('reqfiles/AfterNLP.csv',index=False)

In [21]:
import os

os.makedirs("models", exist_ok=True)
w2v_model.save("models/w2v_model.model")